# Blog 06 — Why Does a Neuron Need an Activation Function?

> 🧪 **[Open in Google Colab](https://colab.research.google.com/github/manish7725/deeplearning/blob/reorg/class8-to-phd-curriculum/notebooks/06-why-neurons-need-activation.ipynb)**

📖 **[Read the matching blog](https://github.com/manish7725/deeplearning/blob/reorg/class8-to-phd-curriculum/blogs/06-why-neurons-need-activation.md)**

Core idea: affine transformations alone stay affine; nonlinear activations let composition become richer.

## 🧭 Lesson bridge

**Came from:** Blog 05 — $z=w^Tx+b$.

**Today:** $a=f(z)$ adds nonlinearity.

**Next:** Blog 07 — prediction is not learning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)


## 1. Prove the linear-layer problem

Compose `f(x)=2x+1` and `g(x)=3x-4`. Algebra predicts `g(f(x))=6x-1`. Verify it numerically.

In [ ]:
x = np.linspace(-3, 3, 100)
two_layers = 3*(2*x + 1) - 4
collapsed = 6*x - 1
assert np.allclose(two_layers, collapsed)
plt.plot(x, two_layers, label="two affine layers")
plt.plot(x, collapsed, '--', label="one affine function")
plt.xlabel("x"); plt.ylabel("output"); plt.legend()
plt.title("Stacking affine functions does not create a bend")
plt.show()


## 2. ReLU by hand

$$ReLU(z)=\max(0,z).$$

Predict the result for `[-3,-1,0,2,5]` before running the code.

In [ ]:
z = np.array([-3., -1., 0., 2., 5.])
relu = np.maximum(0, z)
print(relu)
assert np.allclose(relu, [0.,0.,0.,2.,5.])


## 3. Compare activations

We compare shape, output range, and saturation behavior. ReLU is piecewise linear; sigmoid and tanh are smooth and saturate at large magnitudes.

In [ ]:
z = np.linspace(-6, 6, 400)
sigmoid = 1/(1+np.exp(-z))
tanh = np.tanh(z)
relu = np.maximum(0, z)
plt.plot(z, sigmoid, label="sigmoid")
plt.plot(z, tanh, label="tanh")
plt.plot(z, relu, label="ReLU")
plt.axhline(0); plt.axvline(0)
plt.xlabel("pre-activation z"); plt.ylabel("activation")
plt.title("Three activation functions"); plt.legend(); plt.show()


## 4. Numerical derivative experiment

A derivative tells us how much the output changes for a tiny input change. Estimate the derivative around zero using a centered difference.

For sigmoid, the exact derivative is `sigmoid(z)*(1-sigmoid(z))`.

For ReLU, the derivative is 0 for negative inputs and 1 for positive inputs; at exactly zero it is not uniquely defined.

In [ ]:
def sigmoid_fn(v):
    return 1/(1+np.exp(-v))

h = 1e-5
z0 = 0.0
numerical = (sigmoid_fn(z0+h)-sigmoid_fn(z0-h))/(2*h)
exact = sigmoid_fn(z0)*(1-sigmoid_fn(z0))
print("numerical derivative =", numerical)
print("exact derivative =", exact)
assert np.isclose(numerical, exact, atol=1e-5)


## 5. See a ReLU bend

A single ReLU unit creates a kink. Several units can create multiple pieces. This is the beginning of piecewise-linear function building.

In [ ]:
g = np.maximum(0, 2*x + 1)
plt.plot(x, g)
plt.axhline(0); plt.axvline(-0.5)
plt.xlabel("x"); plt.ylabel("ReLU(2x+1)")
plt.title("ReLU creates a change of regime at 2x+1=0")
plt.show()


## 6. Interactive animation

Watch a neuron change as its weight changes. Predict what happens to the kink location before running.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

weights = np.linspace(-3, 3, 50)
fig, ax = plt.subplots()
ax.set_xlim(-3, 3); ax.set_ylim(0, 7)
line, = ax.plot([], [])

def draw(i):
    weight = weights[i]
    line.set_data(x, np.maximum(0, weight*x + 1))
    ax.set_title(f"ReLU neuron: weight={weight:.2f}, bias=1")
    return (line,)

ani = FuncAnimation(fig, draw, frames=len(weights), interval=60, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())


## 7. PyTorch bridge

The same functions are available in PyTorch and can participate in automatic differentiation.

In [ ]:
import torch

tz = torch.tensor([-2., -1., 0., 1., 2.])
print("ReLU:", torch.relu(tz))
print("sigmoid:", torch.sigmoid(tz))
print("tanh:", torch.tanh(tz))


## 8. Failure mode experiment

Sigmoid derivatives become very small far from zero. ReLU can have zero derivative on its negative side. Plot the derivatives and connect this to the future problem of gradient flow.

In [ ]:
sigmoid_derivative = sigmoid * (1-sigmoid)
relu_derivative = (z > 0).astype(float)
plt.plot(z, sigmoid_derivative, label="sigmoid derivative")
plt.plot(z, relu_derivative, label="ReLU derivative")
plt.xlabel("z"); plt.ylabel("derivative"); plt.legend()
plt.title("Activation derivatives"); plt.show()


## 9. Challenges

**Beginner:** compute ReLU by hand for five values.

**Builder:** implement Leaky ReLU without using a library activation.

**Scientist:** compare sigmoid, tanh, and ReLU under the same tiny neural network and measure training speed. Keep the seed and all other settings fixed.

**Research bridge:** investigate how activation choice affects gradient propagation, optimization, and final representation quality. Document the controlled variables.

## 🏁 Mastery gate

Explain why affine layers alone collapse, draw ReLU/sigmoid/tanh, compute ReLU by hand, estimate a derivative numerically, and explain one activation failure mode.

**Next:** Blog 07 — Prediction Is Not the Same as Learning.